# Feature engineering and leakage-safe preprocessing

Run from top to bottom. Quick mode is a patient subset, not final results. Notebook outputs are separate from production reports.

In [ ]:
from pathlib import Path
import sys, os
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src'/'features.py').exists()), None)
assert ROOT is not None, 'Open the notebook from the project repository'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.notebook_utils import *
QUICK = True  # False uses the entire eligible dataset; keep consistent across notebooks
OUT = ROOT/'reports'/('notebook_quick' if QUICK else 'notebook_full')
OUT.mkdir(parents=True, exist_ok=True)


Split patients before fitting transformations

In [ ]:
raw, x, y, g, parts = load_cohort(ROOT, QUICK)
train, cal, tune, test = (parts[k] for k in ("train","calibration","threshold","test"))
X, Y, G = x.iloc[train], y.iloc[train], g.iloc[train]
display(split_table(y,g,parts))

Inspect original and engineered features

In [ ]:
from src.features import features, DROP
assert not set(['patient_nbr','encounter_id','readmitted','target']) & set(x.columns)
display(X.head()); display(X.dtypes.value_counts()); display(X.isna().sum().sort_values(ascending=False).head(10))

Fit imputation, scaling, ordinal encoding and one-hot encoding on training only.

In [ ]:
from src.preprocessing import build
ordinal,encoder,cat=build(X)
a=ordinal.fit_transform(X); b=encoder.fit_transform(a)
c=encoder.transform(ordinal.transform(x.iloc[tune]))
print('Train transformed:',b.shape,'Validation transformed:',c.shape,'Categorical positions:',cat)
display(pd.DataFrame(b[:5],columns=encoder.get_feature_names_out()))